In [1]:
import psycopg2
import pandas as pd
from ftplib import FTP
from sqlalchemy import create_engine, text
import os 
import re
from dotenv import load_dotenv
from datetime import datetime

In [3]:
load_dotenv(dotenv_path='env.txt')  # Looks for .env in current dir and loads it


print("POSTGRES_USER:", os.getenv("POSTGRES_USER"))


POSTGRES_USER: doadmin


In [4]:
import logging
from datetime import datetime
import os
import re
import pandas as pd
from ftplib import FTP
from sqlalchemy import create_engine, text

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("ftp_ingestion.log", mode='a')
    ]
)

def check_file_path_timestamp(file_name):
    logging.info(f"Extracting date from file name: {file_name}")
    match = re.search(r'\d{8}', file_name)
    if match:
        date_string = match.group()
        quote_date = pd.to_datetime(date_string, format='%Y%m%d')
        logging.info(f"Parsed quote date: {quote_date}")
        return quote_date
    else:
        logging.warning(f"No date found in file name: {file_name}")
        return None

def download_most_recent_file_locally(ftp, conn, file_name):
    local_directory = os.getcwd()
    try:
        local_path = os.path.join(local_directory, file_name)
        logging.info(f"Downloading file: {file_name} to {local_path}")

        with open(local_path, 'wb') as f:
            ftp.retrbinary('RETR ' + file_name, f.write)

        logging.info(f"Successfully downloaded: {file_name}")
        get_first_row(local_path, conn)
        logging.info(f"Successfully processed: {file_name}")
    except Exception as e:
        logging.error(f"Error downloading or processing {file_name}: {e}")

def get_first_row(file_path, conn):    
    logging.info(f"Parsing file: {file_path}")
    is_xslx = False
    regex = re.compile(r'(xlsx|xls)')

    if regex.search(file_path) and '20YR' not in file_path:
        logging.info("File is Excel format.")
        xlsx = pd.ExcelFile(file_path)
        is_xslx = True
        regex = re.compile(r'(Fixed Prices - ATC)')
        sheet_name = ''
        for name in xlsx.sheet_names:
            if regex.search(name):
                sheet_name = name
                logging.info(f"Found target sheet: {sheet_name}")
                raw_df = pd.read_excel(xlsx, name, header=0)
                break
        else:
            sheet_name = xlsx.sheet_names[0]
            raw_df = pd.read_excel(xlsx)
            logging.info(f"Using default sheet: {sheet_name}")
    else:
        logging.info("File is CSV format.")
        raw_df = pd.read_csv(file_path)

    raw_df.dropna(axis=0, how='all', inplace=True)
    raw_df.dropna(axis=1, how='all', inplace=True)
    quote_date = check_file_path_timestamp(file_name=file_path)

    if is_xslx:
        file_type = 'xls' if re.search(r'(?:.*xls$)', file_path) else 'xlsx'
        if sheet_name == 'Fixed Prices - ATC':
            fixed_atc_parser(raw_df, quote_date, file_type, conn)
        elif sheet_name == 'PW_Vol - Mid':
            logging.info("Volatility file found, skipping.")
            return
    else:
        options_parser(raw_df, conn)
        logging.info(f"Removing CSV file after processing: {file_path}")
        os.remove(file_path)

def options_parser(df, conn):
    logging.info("Processing options price file.")
    df.drop(df.columns[0], axis=1, inplace=True)
    df.columns = df.columns.str.lower()
    df = df[df['contract_term'] == 'Month'].reset_index(drop=True)
    df['iso_zone'] = df['iso'] + '_' + df['market']
    df['iso_zone'] = df['iso_zone'].str.upper()
    df = df[df['iso_zone'] == 'NYMEX_NATURAL GAS']

    logging.info(f"Filtered down to {len(df)} rows after ISO filter.")

    df.drop(columns=['iso', 'market', 'market_code', 'contract_end', 'time_key', 'data_code', 'contract_name', 'contract_term', 'atc', 'hr', 'iso_zone', 'bid', 'ask'], inplace=True)
    df.rename(columns={'curve_date': 'quote_date', 'contract_begin': 'start_month', 'mid': 'price'}, inplace=True)
    df = get_mapping_id(df, conn, col='peak_hour', col2='time_id', table='options_peak_hour_mapping')
    df['price'] = df['price'] / 10

    sql_query = "DELETE FROM options_prices WHERE quote_date = CURRENT_DATE - INTERVAL '2 DAY'"
    result = conn.execute(text(sql_query))
    logging.info(f"Deleted {result.rowcount} rows from options_prices to prevent duplication.")

    insert_table(df, 'options_prices', conn)

def volatility_parser(df, quote_date):
    df = df.drop(df.columns[0:2], axis = 1)
    df = df.tail(-2)
    new_column_names = df.iloc[0].tolist()
    df.columns = new_column_names
    df = df.tail(-1)
    df.drop(df.columns[-1], axis=1, inplace=True)
    df = df.melt(id_vars=['Contract Month:', 'Strike:'], 
                    var_name='iso_zone', 
                    value_name='percentage')
    df.rename(columns={'Contract Month:': 'contract_month', 'Strike:': 'strike'}, inplace=True)
    df['quote_date'] = quote_date 
    
    #insert_table(df, 'volatility_percentages')

# transposes df for start months, reformats for postgres db

def fixed_atc_parser(df, quote_date, file_type, conn):
    logging.info("Processing fixed ATC price file.")
    df = df.drop(df.columns[0], axis=1)
    if file_type in ['xls', 'xlsx']:
        df.loc[3] = df.loc[4] + '_' + df.loc[5]

    df = df.T
    df = df.drop(df.columns[1:4], axis=1)
    df.rename(columns={df.columns[0]: 'iso_zone'}, inplace=True)
    df.loc['Unnamed: 1', 'iso_zone'] = 'iso_zone'
    df.columns = df.iloc[0].tolist()
    df = df.tail(-1)

    df = df.melt(id_vars=['iso_zone'], var_name='start_month', value_name='price')
    df['price'] = df['price'] / 1000
    df['quote_date'] = quote_date 
    df['iso_zone'] = df['iso_zone'].str.upper()
    df = get_mapping_id(df, conn, col='iso_zone', col2='id', table='iso_zone_mapping')
    insert_table(df, 'fixed_prices_atc', conn)

def get_mapping_id(df, conn, col, col2, table):
    logging.info(f"Mapping {col} to {col2} from {table}.")
    sql_query = f"SELECT {col}, {col2} FROM {table}"
    result = conn.execute(text(sql_query))
    mapping_df = pd.DataFrame(result.fetchall(), columns=result.keys())
    df = pd.merge(df, mapping_df, how='left', on=col)
    df.drop(columns=col, inplace=True)
    return df

def insert_table(df, source_to_table, conn):
    logging.info(f"Inserting data into table: {source_to_table} ({len(df)} rows)")
    df.to_sql(source_to_table, conn, if_exists='append', index=False)
    conn.commit()
    logging.info("Insert complete.")

def run_locally_on_ftp_server(conn):
    logging.info("📦 Starting FTP ingestion routine...")

    try:
        # Load credentials
        server_address = os.getenv('OTC_FTP_SERVER_ADDRESS')
        username = os.getenv('OTC_FTP_USERNAME')
        password = os.getenv('OTC_FTP_PASSWORD')
        port = int(os.getenv('OTC_FTP_PORT'))

        if not all([server_address, username, password, port]):
            raise EnvironmentError("❌ Missing FTP credentials or configuration.")

        # Format the backfill date
        backfill_date = datetime(2025, 7, 14).date()
        backfill_token = backfill_date.strftime('%Y%m%d')  # '20250714'

        logging.info(f"📅 Looking for files with token: {backfill_token}")

        with FTP() as ftp:
            ftp.connect(server_address, port)
            ftp.login(username, password)
            logging.info("🔐 FTP login successful.")

            files_list = ftp.nlst()
            logging.info(f"📂 Found {len(files_list)} files on FTP.")

            # Find files that match backfill date pattern
            matched_files = [
                f for f in files_list
                if backfill_token in f and f.lower().endswith(('.csv', '.xls', '.xlsx')) and '20YR' not in f
            ]

            if not matched_files:
                logging.warning("⚠️ No files matched for backfill date.")
            else:
                for file_name in matched_files:
                    logging.info(f"✅ Processing matched file: {file_name}")
                    download_most_recent_file_locally(ftp, conn, file_name)

    except Exception as e:
        logging.error(f"❌ Error during FTP ingestion: {e}")
    else:
        logging.info("✅ FTP ingestion routine completed successfully.")




In [5]:

def testing():
    logging.info("Starting local FTP ingestion test.")
    connection_string = "postgresql://" + os.getenv('POSTGRES_USER') + ":" + os.getenv("POSTGRES_PASSWORD") + "@" + os.getenv("POSTGRES_HOST") + ":" + os.getenv("POSTGRES_PORT") + "/" + os.getenv('POSTGRES_DBNAME') + "?sslmode=require"
    engine = create_engine(connection_string)
    conn = engine.connect()
    run_locally_on_ftp_server(conn)
    conn.close()
    logging.info("Local FTP ingestion test complete.")

In [6]:
if __name__ == "__main__":
    testing()


2025-07-16 16:31:52,617 [INFO] Starting local FTP ingestion test.
2025-07-16 16:31:53,168 [INFO] 📦 Starting FTP ingestion routine...
2025-07-16 16:31:53,170 [INFO] 📅 Looking for files with token: 20250714
2025-07-16 16:31:53,297 [INFO] 🔐 FTP login successful.
2025-07-16 16:31:53,451 [INFO] 📂 Found 15129 files on FTP.
2025-07-16 16:31:53,454 [INFO] ✅ Processing matched file: EOD_CORR_20250714_1430.xlsx
2025-07-16 16:31:53,455 [INFO] Downloading file: EOD_CORR_20250714_1430.xlsx to /Users/muhammadsubhanmunir/EOD_CORR_20250714_1430.xlsx
2025-07-16 16:31:53,564 [INFO] Successfully downloaded: EOD_CORR_20250714_1430.xlsx
2025-07-16 16:31:53,565 [INFO] Parsing file: /Users/muhammadsubhanmunir/EOD_CORR_20250714_1430.xlsx
2025-07-16 16:31:53,566 [INFO] File is Excel format.
2025-07-16 16:31:53,704 [INFO] Using default sheet: Power - NYMEX Corr.
2025-07-16 16:31:53,705 [INFO] Extracting date from file name: /Users/muhammadsubhanmunir/EOD_CORR_20250714_1430.xlsx
2025-07-16 16:31:53,706 [INFO] Pa